# Data Manipulation, Cleaning and Imputation 🧹

### ISA 383: Python for Business Analytics

Real data is rarely ready for analysis.
This notebook follows one repeatable workflow on a messy copy of the *Titanic* passenger list: inspect the data, diagnose each problem, decide what to do, apply the change, and validate the result.
Along the way you will convert data types, handle missing values, and deal with outliers.

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.

## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Diagnose common data-quality problems: messy column names, duplicates, inconsistent labels, wrong types, missing values, and outliers.
2. Convert columns to appropriate types with `pd.to_numeric()`, `pd.to_datetime()`, `astype()`, nullable integers, and `category`, and verify each conversion.
3. Explain why values are missing, and choose between dropping, flagging, and imputing them, including group-wise imputation with `transform()`.
4. Detect outliers with the IQR and z-score rules, and decide whether to keep, flag, cap, or transform them.
5. Package cleaning steps into a reusable function and validate its output with assertions.

## 1. The story: a second copy of the passenger list 🗂️

In the weeks after the *Titanic* sank on 15 April 1912, lists of passengers were compiled, copied, and re-copied by shipping-line clerks, newspapers, and inquiry staff.
Every copy introduced small inconsistencies: abbreviations, different date styles, numbers written with currency signs, and the same person entered twice.

To practise, we use `data/titanic_messy.csv`, a **teaching copy** of our passenger file into which we deliberately introduced these typical transcription problems.
The problems we added are not claims about the historical records.
Because we also have the source (`data/titanic.csv`), we can check at the end whether our cleaning recovered the truth, a luxury real analysts rarely have.

The source file has real problems of its own that no clerk introduced: 177 missing ages, 687 missing cabins, two missing ports, fares of £0, and fares shared by everyone on a ticket.
Those are the problems that require genuine judgment.

We will follow the same five steps for every problem:

| Step | Question |
|---|---|
| **Inspect** | What does the data look like? |
| **Diagnose** | What exactly is wrong, and how much of it is there? |
| **Decide** | What should we do, and what assumption does that require? |
| **Apply** | Make the change in code, never by hand. |
| **Validate** | Did the change do what we intended, and nothing else? |

## 2. Setup 🧰

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from urllib.request import urlretrieve

DATA_DIR = Path("/content/isa383/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the course data from the course GitHub repository (only the first time).
DATA_URL = "https://raw.githubusercontent.com/harshvardhaniimi/isa383-google-colab/main/data/"
for filename in ["titanic.csv", "titanic_messy.csv"]:
    if not (DATA_DIR / filename).exists():
        urlretrieve(DATA_URL + filename, DATA_DIR / filename)
print("Course data is ready.")

raw = pd.read_csv(DATA_DIR / "titanic_messy.csv")
source = pd.read_csv(DATA_DIR / "titanic.csv")   # used only to check our work at the end

print("Messy copy:", raw.shape, "| source file:", source.shape)

Course data is ready.
Messy copy: (897, 13) | source file: (891, 12)


## 3. Diagnose before changing anything 🔎

A quality check is not only a hunt for errors.
It is a compact description of what the file contains and where decisions are needed.
Resist the urge to fix things as you spot them; first build a complete list of problems.

### 🧭 Syntax first: Diagnostic tools

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.info()
df.isna().sum()
df.nunique(dropna=False)
df["column"].value_counts(dropna=False)
df.duplicated(keep="first")                  # exact duplicate rows
df.duplicated(subset=["key"], keep=False)    # all rows that share a key
```

Documentation: [info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) · [nunique](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html) · [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

In [2]:
raw.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare (GBP),Cabin,Embarked,Boarding Date
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,NaN,S,1912-04-10
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,£71.2833,C85,C,10 April 1912
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,£7.925,NaN,S,1912-04-10
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,£53.1,C123,S,10 April 1912
4,5,0,3rd,"Allen, Mr. William Henry",male,35,0,0,373450,£8.05,NaN,S,10/04/1912


In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 897 entries, 0 to 896
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0    PassengerId   897 non-null    int64 
 1   Survived       897 non-null    object
 2   Pclass         897 non-null    object
 3   Name           897 non-null    object
 4   Sex            897 non-null    object
 5   Age            818 non-null    object
 6   SibSp          897 non-null    int64 
 7   Parch          897 non-null    int64 
 8   Ticket         897 non-null    object
 9   Fare (GBP)     897 non-null    object
 10  Cabin          205 non-null    object
 11  Embarked       895 non-null    object
 12  Boarding Date  895 non-null    object
dtypes: int64(3), object(10)
memory usage: 91.2+ KB


In [4]:
# Column names can hide spaces that make raw["Sex"] fail.
raw.columns.tolist()

[' PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex ',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare (GBP)',
 'Cabin',
 'Embarked',
 'Boarding Date']

In [5]:
def quality_report(df):
    # One row per column: its type, how much is missing, and how varied it is.
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(1),
            "unique_values": df.nunique(dropna=False),
            "example": df.iloc[0],
        }
    )

quality_report(raw)

,dtype,missing,missing_pct,unique_values,example
PassengerId,int64,0,0.0,891,1
Survived,object,0,0.0,4,0
Pclass,object,0,0.0,6,3
Name,object,0,0.0,891,"Braund, Mr. Owen Harris"
Sex,object,0,0.0,10,male
Age,object,79,8.8,92,22
SibSp,int64,0,0.0,7,1
Parch,int64,0,0.0,7,0
Ticket,object,0,0.0,681,A/5 21171
Fare (GBP),object,0,0.0,363,7.25


In [6]:
# Categorical columns should have only a few distinct values. Do they?
for column in ["Survived", "Pclass", "Sex ", "Embarked"]:
    print(column.strip(), "->", raw[column].value_counts(dropna=False).to_dict())
    print()

Survived -> {'0': 526, '1': 331, 'No': 27, 'Yes': 13}

Pclass -> {'3': 463, '1': 202, '2': 172, '3rd': 31, '1st': 15, '2nd': 14}

Sex -> {'male': 525, 'female': 282, 'MALE': 18, 'Male': 14, 'M': 14, ' male': 11, 'Female': 11, 'F': 9, 'female ': 8, 'FEMALE': 5}

Embarked -> {'S': 601, 'C': 150, 'Q': 73, 'S ': 20, 'Southampton': 16, 's': 11, ' C': 9, 'c': 7, 'Cherbourg': 3, 'q': 3, nan: 2, 'Queenstown': 2}



In [7]:
print("Exact duplicate rows:", raw.duplicated().sum())

repeated_ids = raw.duplicated(subset=[" PassengerId"], keep=False)
raw.loc[repeated_ids, [" PassengerId", "Name", "Age", "Fare (GBP)"]].head(6)

Exact duplicate rows: 6


,PassengerId,Name,Age,Fare (GBP)
57,58,"Novel, Mr. Mansouer",28.5,7.2292
58,58,"Novel, Mr. Mansouer",28.5,7.2292
369,369,"Jermyn, Miss. Annie",999,7.75
370,369,"Jermyn, Miss. Annie",999,7.75
698,697,"Kelly, Mr. James",44,8.05
699,697,"Kelly, Mr. James",44,8.05


**Problem log.**

| # | Problem | Planned fix |
|---|---|---|
| 1 | Column names contain spaces, capitals, and symbols | Standardize to `snake_case` |
| 2 | Six rows were entered twice | Drop exact duplicates; check the key is unique |
| 3 | `Sex`, `Embarked`, `Pclass`, `Survived` use several spellings for one value | Map every variant to one label |
| 4 | `Age`, `Fare (GBP)`, `Pclass`, `Survived` are stored as text | Convert to numbers |
| 5 | `Age` uses `unknown`, `?`, and `999` to mean "missing" | Convert those codes to real missing values |
| 6 | `Boarding Date` uses three date formats | Parse each format explicitly and validate |
| 7 | `Age`, `Cabin`, `Embarked` have genuine gaps | Decide column by column (Section 8) |
| 8 | `Fare` is very skewed, with £0 and £512 values | Investigate before treating (Section 9) |

> 💭 **Quick thought question:** Why is checking duplicates on the business key (`PassengerId`) often more informative than checking whether entire rows are identical?

## 4. Clean the column names 🏷️

Consistent, lower-case names without spaces make every later step easier to type and harder to get wrong.
The `.str` accessor works on the column index just as it works on a text column.

### 🧭 Syntax first: Renaming columns systematically

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)   # any run of other characters -> "_"
    .str.strip("_")
)
df = df.rename(columns={"old_name": "new_name"})
```

Documentation: [Working with text](https://pandas.pydata.org/docs/user_guide/text.html) · [rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)

In [8]:
df = raw.copy()   # never modify the raw data in place; keep it for comparison

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
df = df.rename(columns={"passengerid": "passenger_id"})

df.columns.tolist()

['passenger_id',
 'survived',
 'pclass',
 'name',
 'sex',
 'age',
 'sibsp',
 'parch',
 'ticket',
 'fare_gbp',
 'cabin',
 'embarked',
 'boarding_date']

## 5. Remove duplicates 👯

Exact duplicates are usually safe to drop.
Duplicates on a key with *different* values elsewhere are harder: you must decide which record to trust, for example the most recent or the most complete one.

### 🧭 Syntax first: Finding and dropping duplicates

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.duplicated().sum()
df.drop_duplicates()                                  # exact duplicates
df.drop_duplicates(subset=["key"], keep="first")      # "first", "last", or False (drop all)
df["key"].is_unique
```

Documentation: [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [9]:
rows_before = len(df)
df = df.drop_duplicates()

print("Rows removed:", rows_before - len(df))
print("Is passenger_id unique now?", df["passenger_id"].is_unique)

Rows removed: 6
Is passenger_id unique now? True


> 💭 **Quick thought question:** Imagine two rows with the same `passenger_id`, one with an age and one without.
Would `keep="first"` be a good rule?
What rule would you write instead?

## 6. Standardize text labels 🔤

`male`, `Male`, ` male`, `MALE`, and `M` mean the same thing to a person but are five different values to pandas.
Every `groupby()` or filter on such a column would silently split one group into several.
The fix is to normalize the text (strip spaces, change case) and then map the remaining variants explicitly.

### 🧭 Syntax first: Cleaning text values

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
s.str.strip().str.lower()             # also .str.upper(), .str.title()
s.replace({"m": "male", "f": "female"})  # change only the listed values
s.map({"S": "S", "SOUTHAMPTON": "S"})    # values NOT in the dict become NaN
s.str.extract(r"(\d)")[0]             # keep only the part that matches a pattern
```

Documentation: [replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) · [map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html) · [str.extract](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html)

In [10]:
print("Before:", df["sex"].value_counts().to_dict())

df["sex"] = df["sex"].str.strip().str.lower().replace({"m": "male", "f": "female"})

print("After: ", df["sex"].value_counts().to_dict())

Before: {'male': 520, 'female': 281, 'MALE': 18, 'Male': 14, 'M': 14, ' male': 11, 'Female': 11, 'F': 9, 'female ': 8, 'FEMALE': 5}
After:  {'male': 577, 'female': 314}


In [11]:
embarked_map = {
    "S": "S", "SOUTHAMPTON": "S",
    "C": "C", "CHERBOURG": "C",
    "Q": "Q", "QUEENSTOWN": "Q",
}

missing_before = df["embarked"].isna().sum()
df["embarked"] = df["embarked"].str.strip().str.upper().map(embarked_map)
missing_after = df["embarked"].isna().sum()

print(df["embarked"].value_counts(dropna=False).to_dict())
print("New missing values created by map():", missing_after - missing_before)

{'S': 644, 'C': 168, 'Q': 77, nan: 2}
New missing values created by map(): 0


`map()` turns any value that is not in the dictionary into `NaN`.
That is useful, because an unexpected spelling cannot slip through, but it means you must check that no new missing values appeared.
Here the check shows zero, so every variant was covered.

In [12]:
# "1st" -> "1" with a regular expression; "Yes"/"No" -> "1"/"0" with replace().
df["pclass"] = df["pclass"].str.extract(r"(\d)")[0]
df["survived"] = df["survived"].replace({"Yes": "1", "No": "0"})

print(df["pclass"].value_counts().to_dict())
print(df["survived"].value_counts().to_dict())

{'3': 491, '1': 216, '2': 184}
{'0': 549, '1': 342}


### 🧩 Guided practice: Standardize a new column

The cell below creates a small Series of lifeboat-station labels with inconsistent spelling.
Standardize it to exactly three labels, `Port`, `Starboard`, and `Unknown`, and confirm the result with `value_counts()`.

In [13]:
side = pd.Series(["port", "Port ", "STARBOARD", "starbd", " Starboard", None, "P", "S"])

# TODO: Normalize case and spaces, map the variants, and fill the remaining gaps with "Unknown".

## 7. Type conversions 🔁

A column's type decides what you can do with it.
Text that looks like a number cannot be averaged, text sorts alphabetically (so `"100"` comes before `"20"`), and text dates cannot be subtracted.
Conversions should be **explicit** and **checked**: always look at what failed to convert.

### 🧭 Syntax first: Converting types

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
pd.to_numeric(s, errors="coerce")              # unparseable values -> NaN
s.astype("Int64")                              # nullable integer (capital I allows missing values)
s.astype("category")                           # few distinct values, less memory
pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
pd.read_csv(path, na_values={"column": ["unknown", "?"]})   # prevent problems at read time
```

Documentation: [to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [Nullable integers](https://pandas.pydata.org/docs/user_guide/integer_na.html) · [Categorical data](https://pandas.pydata.org/docs/user_guide/categorical.html)

In [14]:
# Text that looks like a number still behaves like text.
try:
    df["fare_gbp"].mean()
except TypeError as error:
    print("TypeError:", error)

print("Sorted as text:", sorted(["100", "20", "3"]))

TypeError: Could not convert string '7.25£71.2833£7.925£53.1£8.058.458351.862521.075£11.133330.070816.7£26.558.0531.2757.854216£29.125£1318£7.22526138.029235.5£21.075£31.38757.225263£7.87927.895827.7208146.52087.7510.582.1708£527.22928.051811.2417£9.475217.8958£41.57927.87928.0515.57.7521.6792£17.839.68757.876.7292£2661.979235.5£10.57.2292£27.7546.9£7.229280£83.475£27.927.720815.245810.58.15837.9258.662510.546.973.5 14.4542 56.49587.657.89588.052912.47599.57.787547.1£10.515.8534.3758.052638.05£8.057.8542£61.175£20.575£7.25£8.0534.654263.358323267.8958 7.8958 £77.28758.65427.9257.89587.657.7757.8958£24.15£5214.45428.059.82514.45837.9257.75£21£247.5208£31.27573.58.0530.07081377.2875£11.24177.75 7.1417 22.3583£6.9757.89587.05£14.5261315.045826.283353.19.216779.215.24587.75£15.85£6.75£11.5£36.757.795834.375 26 13£12.52566.68.0514.57.312561.37927.73338.058.662569.5516.1£15.757.7758.6625£39.687520.52555£27.925.925£56.495833.529.125£11.13337.92530.6958£7.854225.4667£28.7125130 £69.55 15.05 31

In [15]:
# Fares: remove the currency sign and spaces, then convert.
fare_text = df["fare_gbp"].str.replace("£", "", regex=False).str.strip()
df["fare_gbp"] = pd.to_numeric(fare_text, errors="coerce")

print("Fares that failed to convert:", df["fare_gbp"].isna().sum())
df["fare_gbp"].describe().round(2)

Fares that failed to convert: 0


count    891.00
mean      32.20
std       49.69
min        0.00
25%        7.91
50%       14.45
75%       31.00
max      512.33
Name: fare_gbp, dtype: float64

In [16]:
# Ages: errors="coerce" turns anything unparseable into NaN. Inspect exactly what was lost.
age_numeric = pd.to_numeric(df["age"], errors="coerce")

lost = age_numeric.isna() & df["age"].notna()
print("Values coerced to NaN:", df.loc[lost, "age"].value_counts().to_dict())

age_numeric.describe().round(1)

Values coerced to NaN: {'unknown': 42, '?': 25}


count    745.0
mean      70.0
std      194.2
min        0.4
25%       21.0
50%       29.0
75%       40.0
max      999.0
Name: age, dtype: float64

`unknown` and `?` became `NaN`, as intended.
But the maximum age is 999: a **sentinel value**, a code someone typed to mean "unknown."
It passed the type conversion because it *is* a number, which is why type checks alone are not enough.
A domain rule catches it: no passenger can be older than about 100.

In [17]:
# Keep ages that are plausible; everything else becomes missing.
df["age"] = age_numeric.where(age_numeric.between(0, 100))

print("Missing ages after cleaning:", df["age"].isna().sum())
print("Maximum age now:", df["age"].max())

Missing ages after cleaning: 177
Maximum age now: 80.0


In [18]:
# Prevention is better than cure: na_values can catch these codes while reading.
pd.read_csv(
    DATA_DIR / "titanic_messy.csv",
    na_values={"Age": ["unknown", "?", "999"]},
)["Age"].describe().round(1)

count    719.0
mean      29.8
std       14.5
min        0.4
25%       20.8
50%       28.0
75%       38.0
max       80.0
Name: Age, dtype: float64

In [19]:
# Nullable integers: "Int64" (capital I) stores whole numbers AND allows missing values.
df["survived"] = pd.to_numeric(df["survived"]).astype("Int64")
df["pclass"] = pd.to_numeric(df["pclass"]).astype("Int64")

# Category: a good type for text with few distinct values.
memory_before = df["sex"].memory_usage(deep=True)
df["sex"] = df["sex"].astype("category")
df["embarked"] = df["embarked"].astype("category")
memory_after = df["sex"].memory_usage(deep=True)

print(f"Memory for sex: {memory_before:,} bytes as text -> {memory_after:,} bytes as category")

Memory for sex: 54,979 bytes as text -> 8,235 bytes as category


### Dates: a cautionary tale 📅

The `boarding_date` column mixes three formats: `1912-04-10`, `10/04/1912`, and `10 April 1912`.
pandas can guess the format row by row with `format="mixed"`.
Let's see what happens.

In [20]:
guessed = pd.to_datetime(df["boarding_date"], format="mixed")
guessed.value_counts(dropna=False)

boarding_date
1912-04-10    546
1912-10-04    266
1912-04-11     60
1912-11-04     17
NaT             2
Name: count, dtype: int64

Every passenger boarded on 10 or 11 April 1912, yet the guess produced dates in **October** and **November**.
When pandas sees `10/04/1912`, it assumes the American month-first order and reads 4 October; in the UAE and the UK, the same text means 10 April.
No error was raised; only a validation check against what we know (the voyage dates) reveals the problem.

Passing `dayfirst=True` would fix this particular file, but guessing formats row by row stays fragile, and the guess can change between pandas versions.
The safe approach is to parse each known format explicitly and combine the results.

In [21]:
date_text = df["boarding_date"]

iso = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")        # 1912-04-10
day_first = pd.to_datetime(date_text, format="%d/%m/%Y", errors="coerce")  # 10/04/1912
long_form = pd.to_datetime(date_text, format="%d %B %Y", errors="coerce")  # 10 April 1912

df["boarding_date"] = iso.fillna(day_first).fillna(long_form)
df["boarding_date"].value_counts(dropna=False)

boarding_date
1912-04-10    812
1912-04-11     77
NaT             2
Name: count, dtype: int64

In [22]:
# Validate against domain knowledge: the ship's passenger calls were on 10 and 11 April 1912.
voyage_start, voyage_end = pd.Timestamp("1912-04-10"), pd.Timestamp("1912-04-11")
assert df["boarding_date"].dropna().between(voyage_start, voyage_end).all()

# Dates now support arithmetic: days at sea before the collision on 14 April.
days_at_sea = (pd.Timestamp("1912-04-14") - df["boarding_date"]).dt.days
days_at_sea.value_counts(dropna=False)

boarding_date
4.0    812
3.0     77
NaN      2
Name: count, dtype: int64

In [23]:
df.dtypes

passenger_id              int64
survived                  Int64
pclass                    Int64
name                     object
sex                    category
age                     float64
sibsp                     int64
parch                     int64
ticket                   object
fare_gbp                float64
cabin                    object
embarked               category
boarding_date    datetime64[ns]
dtype: object

> 💭 **Quick thought question:** `errors="coerce"` never raises an error.
Why is that both convenient and dangerous, and what should you always do right after using it?

### 🧩 Guided practice: Convert and verify

The Series below contains ticket prices typed by different clerks.
Convert it to numbers, list the raw values that failed to convert, and decide how to handle each one.

In [24]:
prices = pd.Series(["£7.25", "71.28", " 8.05 ", "free", "£512.33", "N/A", "26"])

# TODO: Remove symbols and spaces, convert with errors="coerce", and inspect what failed.

## 8. Missing data: a decision, not a reflex 🕳️

Dropping a row, filling a value, and leaving a value missing are three different decisions.
The right one depends on **why** the value is missing and on how the result will be used.

Statisticians distinguish three situations:

| Mechanism | Plain-language meaning | Example | Consequence |
|---|---|---|---|
| Missing completely at random (MCAR) | Missingness has nothing to do with anything in the data | A coffee stain blots out random entries | Dropping rows loses data but does not bias results |
| Missing at random (MAR) | Missingness depends on *other recorded* variables | Ages are missing more often for third-class passengers | Use those variables to impute (for example, by group) |
| Missing not at random (MNAR) | Missingness depends on the missing value itself | Customers with very high incomes skip the income question | No simple fix; state the limitation clearly |

You can never fully prove which mechanism applies, but you can look for evidence.

### 🧭 Syntax first: Measuring and handling missing values

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.isna().sum(), df.isna().mean()
df.groupby("group")["column"].apply(lambda s: s.isna().mean())

df.dropna(subset=["column"])                     # drop rows missing this column
df.dropna(thresh=n)                              # keep rows with at least n non-missing values
df.fillna({"column": value})                     # constant, mode, median, ...
df.groupby("group")["column"].transform("median")  # one fill value per row, by group
s.ffill(), s.interpolate()                       # ordered data such as time series
```

Documentation: [Missing data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html) · [dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) · [fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) · [interpolate](https://pandas.pydata.org/docs/reference/api/pandas.Series.interpolate.html)

In [25]:
missing_summary = pd.DataFrame({"missing": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(1)})
missing_summary[missing_summary["missing"] > 0]

,missing,missing_pct
age,177,19.9
cabin,687,77.1
embarked,2,0.2
boarding_date,2,0.2


In [26]:
# Evidence about the mechanism: is missingness related to other variables?
df.groupby("pclass").agg(
    missing_age=("age", lambda s: s.isna().mean()),
    missing_cabin=("cabin", lambda s: s.isna().mean()),
).round(3)

,missing_age,missing_cabin
pclass,,
1,0.139,0.185
2,0.060,0.913
3,0.277,0.976


In [27]:
# Is it related to the outcome we care about?
df.groupby(df["age"].isna())["survived"].mean().rename(index={False: "age recorded", True: "age missing"}).round(3)

age
age recorded    0.406
age missing     0.294
Name: survived, dtype: Float64

Ages are missing about twice as often in third class as in first class, and passengers with a missing age survived less often (29.4% versus 40.6%).
So the ages are **not** missing completely at random.
If we simply dropped those rows, the survival rate would rise from 38.4% to 40.6%: dropping is itself a decision with consequences.

### Strategy 1: drop rows (only when few, and random) ✂️

In [28]:
complete_age = df.dropna(subset=["age"])
print(f"Rows: {len(df)} -> {len(complete_age)}")
print(f"Survival rate: {df['survived'].mean():.3f} -> {complete_age['survived'].mean():.3f}")

Rows: 891 -> 714
Survival rate: 0.384 -> 0.406


### Strategy 2: turn a mostly-empty column into something useful 🏷️

`cabin` is missing for 77% of passengers, mostly in second and third class, so filling it would mean inventing data.
Instead we keep the information that *does* exist: whether a cabin was recorded, and the deck letter (the first character of the cabin number).

In [29]:
df["has_cabin"] = df["cabin"].notna()
df["deck"] = df["cabin"].str[0].fillna("Unknown")

df.groupby("has_cabin")["survived"].agg(["size", "mean"]).round(3)

,size,mean
has_cabin,,
False,687,0.3
True,204,0.667


### Strategy 3: fill with a constant or the most common value 🎯

Only two values of `embarked` are missing.
The most common port (the mode) is Southampton, and *Encyclopedia Titanica* records that both passengers, Amelie Icard and Martha Stone, did board at Southampton.
When outside evidence exists, cite it; when it does not, the mode is a reasonable default for a categorical variable with very few gaps.

In [30]:
most_common_port = df["embarked"].mode()[0]
print("Most common port:", most_common_port)

df["embarked"] = df["embarked"].fillna(most_common_port)

# The boarding date follows from the port.
port_dates = {"S": "1912-04-10", "C": "1912-04-10", "Q": "1912-04-11"}
df["boarding_date"] = df["boarding_date"].fillna(pd.to_datetime(df["embarked"].astype(str).map(port_dates)))

print("Missing ports:", df["embarked"].isna().sum(), "| missing boarding dates:", df["boarding_date"].isna().sum())

Most common port: S
Missing ports: 0 | missing boarding dates: 0


### Strategy 4: fill with a single statistic, and see what it does to the data 📉

In [31]:
overall_median_age = df["age"].median()
age_median_filled = df["age"].fillna(overall_median_age)

print("Overall median age:", overall_median_age)
print("Passengers now aged exactly 28:", (age_median_filled == 28).sum(), "(was", (df["age"] == 28).sum(), ")")
print("Standard deviation:", round(df["age"].std(), 2), "->", round(age_median_filled.std(), 2))

Overall median age: 28.0
Passengers now aged exactly 28: 202 (was 25 )
Standard deviation: 14.53 -> 13.02


Filling 177 gaps with one number creates an artificial spike at age 28 and shrinks the spread of ages.
Worse, it gives every missing boy (title "Master") the age of an adult.

### Strategy 5: fill by group, using what we know about each passenger 👪

The title inside each name carries age information: "Master" was used for boys, and "Miss" often for unmarried, younger women.
Group-wise imputation fills each gap with the median age of passengers who share the same title.

In [32]:
df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.")[0]

# Fold rare titles into broader groups so every group has enough people.
title_groups = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
df["title_group"] = df["title"].replace(title_groups)
df.loc[~df["title_group"].isin(["Mr", "Mrs", "Miss", "Master"]), "title_group"] = "Other"

df.groupby("title_group").agg(
    passengers=("age", "size"),
    missing_age=("age", lambda s: s.isna().sum()),
    median_age=("age", "median"),
)

,passengers,missing_age,median_age
title_group,,,
Master,40,4,3.5
Miss,185,36,21.0
Mr,517,119,30.0
Mrs,126,17,35.0
Other,23,1,48.5


In [33]:
# Flag BEFORE filling, so the information "this value was imputed" is never lost.
df["age_imputed"] = df["age"].isna()

median_by_title = df.groupby("title_group")["age"].transform("median")
df["age"] = df["age"].fillna(median_by_title)

print("Missing ages left:", df["age"].isna().sum())
df.loc[df["age_imputed"], ["name", "title_group", "age"]].head(6)

Missing ages left: 0


,name,title_group,age
5,"Moran, Mr. James",Mr,30.0
17,"Williams, Mr. Charles Eugene",Mr,30.0
19,"Masselmani, Mrs. Fatima",Mrs,35.0
26,"Emir, Mr. Farred Chehab",Mr,30.0
28,"O'Dwyer, Miss. Ellen ""Nellie""",Miss,21.0
29,"Todoroff, Mr. Lalio",Mr,30.0


In [34]:
# Compare the strategies on the same summary statistics.
original_age = df["age"].where(~df["age_imputed"])

comparison = pd.DataFrame(
    {
        "drop missing": original_age.dropna().describe(),
        "overall median": original_age.fillna(overall_median_age).describe(),
        "median by title": df["age"].describe(),
    }
).loc[["count", "mean", "std", "min", "50%", "max"]].round(2)
comparison

,drop missing,overall median,median by title
count,714.00,891.00,891.00
mean,29.70,29.36,29.39
std,14.53,13.02,13.27
min,0.42,0.42,0.42
50%,28.00,28.00,30.00
max,80.00,80.00,80.00


No strategy recovers the true ages; each one is an assumption.
The title-based fill keeps the spread closer to the original and gives boys child-like ages, so it is the most defensible of the three here.
Whatever you choose, keep the `age_imputed` flag and check whether your conclusions change under a different choice.

### Ordered data: forward fill and interpolation ⏩

For data ordered in time, a gap can often be filled from its neighbours.
The tiny ticket-sales series below is invented purely to show the mechanics.

In [35]:
daily_sales = pd.Series(
    [120, None, None, 150, None, 170],
    index=pd.date_range("2026-10-01", periods=6, freq="D"),
    name="tickets_sold",
)

pd.DataFrame(
    {
        "raw": daily_sales,
        "ffill": daily_sales.ffill(),            # carry the last known value forward
        "interpolate": daily_sales.interpolate(),  # draw a straight line between known values
    }
)

,raw,ffill,interpolate
2026-10-01,120.0,120.0,120.0
2026-10-02,NaN,120.0,130.0
2026-10-03,NaN,120.0,140.0
2026-10-04,150.0,150.0,150.0
2026-10-05,NaN,150.0,160.0
2026-10-06,170.0,170.0,170.0


> 💭 **Quick thought question:** For which business data would forward fill be sensible (for example, a product's list price), and for which would it be misleading (for example, daily sales)?

## 9. Outliers: flag first, investigate second ⚠️

An outlier is an observation far from most others.
It may be an **error** (an age of 999), or it may be **real and important** (a £512 suite).
Statistical rules help you find candidates; domain knowledge decides what they are.
Never delete an observation only because a rule flagged it.

### 🧭 Syntax first: Detecting and treating outliers

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
is_outlier = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)      # IQR (box-plot) rule

z = (s - s.mean()) / s.std()
is_outlier = z.abs() > 3                                      # z-score rule

s.clip(lower=low, upper=high)      # cap extreme values (winsorizing)
np.log1p(s)                        # compress a long right tail; log(1 + x) handles zeros
```

Documentation: [quantile](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html) · [clip](https://pandas.pydata.org/docs/reference/api/pandas.Series.clip.html) · [skew](https://pandas.pydata.org/docs/reference/api/pandas.Series.skew.html)

In [36]:
df["fare_gbp"].describe().round(2)

count    891.00
mean      32.20
std       49.69
min        0.00
25%        7.91
50%       14.45
75%       31.00
max      512.33
Name: fare_gbp, dtype: float64

In [37]:
def iqr_outlier_flags(values, k=1.5):
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (values < q1 - k * iqr) | (values > q3 + k * iqr)


def zscore_outlier_flags(values, threshold=3):
    z = (values - values.mean()) / values.std()
    return z.abs() > threshold


fare = df["fare_gbp"]
print("IQR rule flags:    ", iqr_outlier_flags(fare).sum(), "fares")
print("z-score rule flags:", zscore_outlier_flags(fare).sum(), "fares")

IQR rule flags:     116 fares
z-score rule flags: 20 fares


The two rules disagree sharply: the IQR rule flags 116 fares (13% of passengers), while the z-score rule flags 20.
Fares are heavily right-skewed: most tickets were cheap, and a few first-class tickets were very expensive.
On such data the IQR rule flags many perfectly ordinary first-class fares, and the z-score rule is itself distorted because the extreme values inflate the mean and standard deviation.
Comparing like with like, within each class, is more sensible.

In [38]:
# Group-aware rule: compare each fare with the fares in its own class.
df["fare_outlier_in_class"] = df.groupby("pclass")["fare_gbp"].transform(iqr_outlier_flags)

df.groupby("pclass")["fare_outlier_in_class"].sum()

pclass
1    20
2     7
3    52
Name: fare_outlier_in_class, dtype: int64

In [39]:
(
    df.loc[df["fare_outlier_in_class"], ["name", "pclass", "ticket", "fare_gbp", "sibsp", "parch"]]
    .sort_values(["pclass", "fare_gbp"], ascending=[True, False])
    .head(8)
)

,name,pclass,ticket,fare_gbp,sibsp,parch
259,"Ward, Miss. Anna",1,PC 17755,512.3292,0,0
681,"Cardeza, Mr. Thomas Drake Martinez",1,PC 17755,512.3292,0,1
741,"Lesurer, Mr. Gustave J",1,PC 17755,512.3292,0,0
27,"Fortune, Mr. Charles Alexander",1,19950,263.0000,3,2
89,"Fortune, Miss. Mabel Helen",1,19950,263.0000,3,2
342,"Fortune, Miss. Alice Elizabeth",1,19950,263.0000,3,2
440,"Fortune, Mr. Mark",1,19950,263.0000,1,4
312,"Ryerson, Miss. Emily Borie",1,PC 17608,262.3750,2,2


Most third-class "outliers" are large families on a single ticket, such as the Sage and Goodwin families, whose fare covered everyone.
They are not errors; the fare simply belongs to the ticket, as the Advanced notebook showed.
The £0 fares are not errors either: they belong to shipyard and shipping-line staff.
Our decision is therefore to **keep** every fare and **flag** the unusual ones.

In [40]:
# If a later model is sensitive to extreme values, two common treatments are capping and log-transforming.
low, high = df["fare_gbp"].quantile([0.01, 0.99])
fare_capped = df["fare_gbp"].clip(lower=low, upper=high)
fare_log = np.log1p(df["fare_gbp"])

fare_versions = pd.DataFrame(
    {"original": df["fare_gbp"], "capped at 1st/99th pct": fare_capped, "log(1 + fare)": fare_log}
)
fare_summary = fare_versions.describe().T
fare_summary["skewness"] = fare_versions.skew()   # 0 = symmetric; large positive = long right tail
fare_summary.round(2)

,count,mean,std,min,25%,50%,75%,max,skewness
original,891.0,32.20,49.69,0.0,7.91,14.45,31.00,512.33,4.79
capped at 1st/99th pct,891.0,31.22,42.52,0.0,7.91,14.45,31.00,249.01,3.11
log(1 + fare),891.0,2.96,0.97,0.0,2.19,2.74,3.47,6.24,0.39


| Situation | Example | Treatment |
|---|---|---|
| Impossible value | Age 999 | Treat as missing (done in Section 7) |
| Real but rare value | £512 suite, £0 staff fares | Keep and flag; report it |
| Real but distorts a mean-based method | Fares in a regression model | Cap, log-transform, or use the median |
| Unclear | A value nobody can explain | Investigate the source; document the decision |

> 💭 **Quick thought question:** A manager asks you to "remove the outliers" from the fare data before computing average revenue per passenger.
What would you ask before doing it, and what would the £512 suite teach you about the business?

## 10. Data manipulation: deriving useful columns 🛠️

Cleaning often blends into **feature engineering**: creating new columns that make the next analysis possible.
You have already created `title`, `has_cabin`, `deck`, and `age_imputed`.
Two more tools are worth knowing: `np.select()` for rule-based labels, and `apply()` for logic that has no vectorized equivalent.

### 🧭 Syntax first: Rule-based columns

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
np.where(condition, value_if_true, value_if_false)
np.select([condition_1, condition_2], [label_1, label_2], default="other")
df.apply(row_function, axis=1)      # slow: one Python call per row
```

Documentation: [numpy.select](https://numpy.org/doc/stable/reference/generated/numpy.select.html) · [DataFrame.apply](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)

In [41]:
# A "lifeboat priority" label that follows the 1912 custom: women first, then children, then men.
conditions = [
    df["sex"] == "female",
    df["age"] < 13,
]
df["priority_group"] = np.select(conditions, ["woman", "child"], default="man")

df.groupby("priority_group")["survived"].agg(["size", "mean"]).round(3)

,size,mean
priority_group,,
child,41,0.561
man,536,0.16
woman,314,0.742


In [42]:
import time

def priority_label(row):
    if row["sex"] == "female":
        return "woman"
    if row["age"] < 13:
        return "child"
    return "man"

start = time.perf_counter()
labels_apply = df.apply(priority_label, axis=1)
apply_seconds = time.perf_counter() - start

start = time.perf_counter()
labels_select = np.select(conditions, ["woman", "child"], default="man")
select_seconds = time.perf_counter() - start

print("Same result?", (labels_apply.to_numpy() == labels_select).all())
print(f"apply(): {apply_seconds * 1000:.1f} ms | np.select(): {select_seconds * 1000:.2f} ms")

Same result? True
apply(): 2.8 ms | np.select(): 0.07 ms


Both give the same answer, but the vectorized version is many times faster; the gap grows with the number of rows.
Use `apply()` only when no vectorized method exists.

## 11. A reusable, validated cleaning pipeline 🧰

Cleaning by hand in many cells is fine for exploring, but the final version should be a **function**: it documents every decision in one place, runs the same way on next month's file, and can be reviewed by someone else.
Assertions then check that the output meets the rules we expect.

### 🧭 Syntax first: Assertions

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
assert condition, "message shown if the condition is False"
assert df["key"].is_unique
assert df["column"].between(low, high).all()
assert set(df["column"].unique()) <= {"allowed", "values"}
```

Documentation: [assert statement](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement)

In [43]:
EMBARKED_MAP = {"S": "S", "SOUTHAMPTON": "S", "C": "C", "CHERBOURG": "C", "Q": "Q", "QUEENSTOWN": "Q"}
PORT_DATES = {"S": "1912-04-10", "C": "1912-04-10", "Q": "1912-04-11"}


def clean_titanic(raw_df):
    out = raw_df.copy()

    # 1. Column names
    out.columns = (
        out.columns.str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    )
    out = out.rename(columns={"passengerid": "passenger_id"})

    # 2. Duplicates
    out = out.drop_duplicates()

    # 3. Labels
    out["sex"] = out["sex"].str.strip().str.lower().replace({"m": "male", "f": "female"})
    out["embarked"] = out["embarked"].str.strip().str.upper().map(EMBARKED_MAP)
    out["pclass"] = out["pclass"].str.extract(r"(\d)")[0]
    out["survived"] = out["survived"].replace({"Yes": "1", "No": "0"})

    # 4. Types (age codes such as 999 become missing)
    out["fare_gbp"] = pd.to_numeric(out["fare_gbp"].str.replace("£", "", regex=False).str.strip(), errors="coerce")
    age = pd.to_numeric(out["age"], errors="coerce")
    out["age"] = age.where(age.between(0, 100))
    out["survived"] = pd.to_numeric(out["survived"]).astype("Int64")
    out["pclass"] = pd.to_numeric(out["pclass"]).astype("Int64")
    dates = out["boarding_date"]
    out["boarding_date"] = (
        pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
        .fillna(pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce"))
        .fillna(pd.to_datetime(dates, format="%d %B %Y", errors="coerce"))
    )

    # 5. Missing values
    out["embarked"] = out["embarked"].fillna("S")   # Encyclopedia Titanica: both boarded at Southampton
    out["boarding_date"] = out["boarding_date"].fillna(pd.to_datetime(out["embarked"].map(PORT_DATES)))
    out["has_cabin"] = out["cabin"].notna()
    out["deck"] = out["cabin"].str[0].fillna("Unknown")
    out["title"] = out["name"].str.extract(r",\s*([^\.]+)\.")[0]
    out["title_group"] = out["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    out.loc[~out["title_group"].isin(["Mr", "Mrs", "Miss", "Master"]), "title_group"] = "Other"
    out["age_imputed"] = out["age"].isna()
    out["age"] = out["age"].fillna(out.groupby("title_group")["age"].transform("median"))

    # 6. Outlier flags (kept, not removed)
    out["fare_outlier_in_class"] = out.groupby("pclass")["fare_gbp"].transform(iqr_outlier_flags)

    # 7. Final types and order
    out["sex"] = out["sex"].astype("category")
    out["embarked"] = out["embarked"].astype("category")
    return out.sort_values("passenger_id").reset_index(drop=True)


clean = clean_titanic(raw)
clean.head()

,passenger_id,survived,pclass,name,sex,age,sibsp,parch,ticket,fare_gbp,cabin,embarked,boarding_date,has_cabin,deck,title,title_group,age_imputed,fare_outlier_in_class
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1912-04-10,False,Unknown,Mr,Mr,False,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1912-04-10,True,C,Mrs,Mrs,False,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1912-04-10,False,Unknown,Miss,Miss,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1912-04-10,True,C,Mrs,Mrs,False,False
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1912-04-10,False,Unknown,Mr,Mr,False,False


In [44]:
# Validation rules every clean file must satisfy.
assert len(clean) == 891, "expected one row per passenger"
assert clean["passenger_id"].is_unique
assert set(clean["sex"].unique()) == {"male", "female"}
assert set(clean["embarked"].unique()) == {"S", "C", "Q"}
assert clean["pclass"].isin([1, 2, 3]).all()
assert clean["age"].between(0, 100).all() and clean["age"].notna().all()
assert clean["fare_gbp"].ge(0).all()
assert clean["boarding_date"].between(voyage_start, voyage_end).all()

print("All validation checks passed.")

All validation checks passed.


In [45]:
# Because this is a teaching copy, we can compare our result with the source file.
check = clean.merge(source, left_on="passenger_id", right_on="PassengerId", validate="one_to_one")

print("Survived matches:", (check["survived"] == check["Survived"]).all())
print("Pclass matches:  ", (check["pclass"] == check["Pclass"]).all())
print("Sex matches:     ", (check["sex"].astype(str) == check["Sex"]).all())
print("Fare matches:    ", np.allclose(check["fare_gbp"], check["Fare"]))

recorded = ~check["age_imputed"]
print("Recorded ages match:", (check.loc[recorded, "age"] == check.loc[recorded, "Age"]).all())

Survived matches: True
Pclass matches:   True
Sex matches:      True
Fare matches:     True
Recorded ages match: True


In [46]:
# Save the result. CSV is readable anywhere but forgets dtypes; Parquet remembers them.
output_dir = Path("/content/isa383/output")
output_dir.mkdir(parents=True, exist_ok=True)

clean.to_csv(output_dir / "titanic_clean.csv", index=False)
clean.to_parquet(output_dir / "titanic_clean.parquet", index=False)

reread = pd.DataFrame(
    {
        "after cleaning": clean.dtypes.astype(str),
        "re-read from CSV": pd.read_csv(output_dir / "titanic_clean.csv").dtypes.astype(str),
        "re-read from Parquet": pd.read_parquet(output_dir / "titanic_clean.parquet").dtypes.astype(str),
    }
)
reread.loc[["pclass", "sex", "embarked", "boarding_date", "has_cabin"]]

,after cleaning,re-read from CSV,re-read from Parquet
pclass,Int64,int64,Int64
sex,category,object,category
embarked,category,object,category
boarding_date,datetime64[ns],object,datetime64[ns]
has_cabin,bool,bool,bool


The CSV round trip loses the category and nullable-integer types (and dates come back as text unless you pass `parse_dates`), while Parquet keeps them.
The *Handling Big Data with Dask* notebook returns to Parquet as a fast format for large files.

## Comprehensive exercises 📝

Work from `raw` (the messy file) or `clean` as each exercise specifies.
For every cleaning decision, write a one-line reason.
A technically valid transformation can still be a poor analytical decision if its assumption is not stated.

### Exercise 1: A better quality report

Extend `quality_report(df)` so that it also reports, for every column stored as text, the share of non-missing values that `pd.to_numeric(..., errors="coerce")` can convert.
A column where almost every value converts is probably a number stored as text.
Run it on `raw` and list the columns that should be numeric.

In [47]:
# Write your solution here.

### Exercise 2: Cleaning ticket prefixes

The `Ticket` column in the **source** file is genuinely messy: some tickets have a prefix such as `A/5`, `PC`, `STON/O2.`, or `SOTON/OQ`.
Extract the prefix (everything before the final block of digits), standardize it by removing `.` and `/` and converting to upper case, and label tickets without a prefix as `NONE`.
Report the number of passengers and the survival rate for every prefix with at least ten passengers.

In [48]:
# Write your solution here.

### Exercise 3: Do the conclusions depend on the imputation?

Starting from the cleaned ages *before* imputation (`clean["age"].where(~clean["age_imputed"])`), build four versions of the age column: drop missing, overall median, median by `pclass` and `sex`, and median by `title_group`.
For each version, compute (a) the mean age of survivors and of non-survivors, and (b) the survival rate of passengers under 13.
Present the results in one table and state which conclusions are robust and which change.

In [49]:
# Write your solution here.

### Exercise 4: Outliers within groups

Write a function `iqr_flags_by_group(df, value_column, group_column, k=1.5)` that returns a Boolean Series.
Compute `fare_per_person` (fare divided by the number of passengers sharing the ticket) and apply your function to it by `pclass`.
Compare the number of flags with the flags on the raw fare, and decide for the three most extreme flagged passengers whether you would keep, cap, or investigate them.

In [50]:
# Write your solution here.

### Exercise 5: A pipeline for a new messy table

The cell below creates a small, **invented** ticket-office ledger with duplicates, prices stored as text, inconsistent class labels, mixed date formats, and missing values.
Write `clean_ledger(df)`, validate its output with at least five assertions, and list every assumption you made.

In [51]:
ledger = pd.DataFrame(
    {
        " Ticket No": ["T001", "T002", "T002", "T003", "T004", "T005", "T006"],
        "Class": ["1st", "third", "third", "2", "Third", None, "FIRST"],
        "Price": ["£30.00", "£7.75", "£7.75", "13", "unknown", "£8.05", " 26.55 "],
        "Sold On": ["2026-04-01", "02/04/2026", "02/04/2026", "3 April 2026", "2026-04-04", "", "05/04/2026"],
        "Passengers": ["1", "4", "4", "2", "1", "1", "999"],
    }
)
ledger

# Write your solution here.

,Ticket No,Class,Price,Sold On,Passengers
0,T001,1st,£30.00,2026-04-01,1
1,T002,third,£7.75,02/04/2026,4
2,T002,third,£7.75,02/04/2026,4
3,T003,2,13,3 April 2026,2
4,T004,Third,unknown,2026-04-04,1
5,T005,None,£8.05,,1
6,T006,FIRST,26.55,05/04/2026,999


## Summary ✅

- **Inspect, diagnose, decide, apply, validate**: build a problem log before changing anything, and check every change.
- Standardize names and labels first; `map()` with an explicit dictionary makes unexpected values visible as `NaN`.
- Convert types explicitly with `to_numeric()`, `to_datetime()` with known formats, `Int64`, and `category`, and always inspect what failed to convert. Watch for sentinel codes such as 999 and for silent date misreads.
- Missing data is a decision: look for evidence of *why* it is missing, flag imputed values, prefer group-wise imputation over a single global value, and check whether conclusions change.
- Outliers are candidates, not verdicts: flag them with IQR or z-score rules (preferably within comparable groups), investigate, and then keep, cap, or transform them.
- Put the final logic in a function, validate it with assertions, and save typed data in Parquet when the types matter.

**Next:** *Data Visualization* turns the tables from the pandas notebooks into charts with matplotlib and seaborn.